# Day 079 — Exercise 4: The Agent Loop

**What you'll build:** `build_agent_prompt` (assembles one step) and `run_agent` (the loop itself).

**Why it matters:** the loop is the whole difference between an agent and a pipeline. It asks the model for one action, runs it, feeds the result back, and repeats — until the model says *finish* or `max_iterations` runs out. That iteration cap is the safeguard against a confused model looping forever.

In [ ]:
import json

def _make_mock_llm(script):
    """Return an llm_fn(messages) that yields each scripted reply in turn.

    Repeats the last reply once the script is exhausted - handy for testing
    a runaway loop (a model that never says 'finish').
    """
    state = {'i': 0}
    def _fn(messages):
        i = state['i']
        state['i'] = min(i + 1, len(script) - 1)
        return script[i]
    return _fn
import ast
import json
import operator

# ── a safe calculator tool (no eval) ─────────────────────────────────────────
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
    ast.USub: operator.neg, ast.UAdd: operator.pos,
}


def _eval_node(node):
    """Recursively evaluate an arithmetic AST node. Raises on anything unsafe."""
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError("unsupported expression")


def safe_calculate(expression):
    """Evaluate a basic arithmetic expression without eval().

    Supports + - * / ** % and parentheses. Anything else (names, calls,
    attribute access) raises ValueError. This is the safe way to give an
    agent a calculator: never eval() untrusted model output.
    """
    tree = ast.parse(expression, mode="eval")
    return _eval_node(tree.body)


# ── the tool registry ────────────────────────────────────────────────────────
# A tool = {description, parameters, fn}. fn takes an args dict, returns a str.
DEFAULT_TOOLS = {
    "calculator": {
        "description": "Evaluate an arithmetic expression, e.g. 2 * (3 + 4).",
        "parameters": {"expression": "string - the arithmetic to evaluate"},
        "fn": lambda args: str(safe_calculate(args["expression"])),
    },
    "word_count": {
        "description": "Count the words in a piece of text.",
        "parameters": {"text": "string - the text to count words in"},
        "fn": lambda args: str(len(str(args["text"]).split())),
    },
}


def build_tool_descriptions(tools):
    """Render a tool registry as a text block for the prompt."""
    lines = []
    for name, spec in tools.items():
        params = ", ".join(spec.get("parameters", {}))
        lines.append("- " + name + "(" + params + "): " + spec["description"])
    return "\n".join(lines)

# ── parsing messy LLM output ──────────────────────────────────────────────────
def safe_parse_json(text):
    """Extract and parse the first JSON object from messy LLM output.

    LLMs wrap JSON in markdown fences or prose. Instead of fighting that,
    slice from the first '{' to the last '}' and parse that. Returns a dict,
    or None if no valid JSON object is present.
    """
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None


def parse_action(text):
    """Turn raw LLM output into an action dict. NEVER raises.

    Returns one of:
      {"type": "tool",   "tool": name, "args": {...}}
      {"type": "finish", "answer": str}
    If the text is not a valid tool call, it falls back to a finish action
    holding the raw text - so a badly-formatted model reply still terminates
    the loop instead of crashing it.
    """
    data = safe_parse_json(text)
    if not isinstance(data, dict):
        return {"type": "finish", "answer": text.strip()}
    tool = data.get("tool")
    if tool and tool != "finish":
        return {"type": "tool", "tool": tool, "args": data.get("args", {})}
    return {"type": "finish", "answer": data.get("answer", text.strip())}

# ── executing tools + calling the model ──────────────────────────────────────
def execute_tool(action, tools):
    """Run one tool action against the registry. Returns a result string.

    Never raises: an unknown tool or a tool error is returned as text so the
    agent can read it and recover on its next turn.
    """
    name = action.get("tool")
    if name not in tools:
        return "Error: unknown tool " + repr(name) + ". Available: " + ", ".join(tools)
    try:
        return str(tools[name]["fn"](action.get("args", {})))
    except Exception as exc:
        return "Error running " + str(name) + ": " + str(exc)


def call_llm(messages, llm_fn=None):
    """Call the chat model. Inject llm_fn(messages) -> str for testing.

    llm_fn=None uses Ollama (llama3.2). A mock llm_fn lets the whole agent
    run offline with no model - which is how the tests drive the loop.
    """
    if llm_fn is not None:
        return llm_fn(messages)
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]


## Task

1. `build_agent_prompt(task, tools, history) -> list[dict]` — a `system` message describing the tools (use `build_tool_descriptions`) and the JSON action format, and a `user` message with the task plus each past step (`You called: ...` / `Result: ...`). Return `[system_msg, user_msg]`.
2. `run_agent(task, tools=None, llm_fn=None, max_iterations=10) -> dict` — default `tools` to `DEFAULT_TOOLS`; loop `for i in range(max_iterations)`: `build_agent_prompt` → `call_llm` → `parse_action`. On `finish` return `{answer, steps, iterations, stopped: False}`. Otherwise `execute_tool`, append `{action, result}` to history, continue. If the loop ends, return `stopped: True`.

## Your Implementation

In [ ]:
def build_agent_prompt(task, tools, history):
    """Build the [system, user] messages for one step of the loop."""
    raise NotImplementedError

def run_agent(task, tools=None, llm_fn=None, max_iterations=10):
    """Run the agent loop. Returns {answer, steps, iterations, stopped}."""
    raise NotImplementedError


In [ ]:

# ── the agent loop ────────────────────────────────────────────────────────────
def build_agent_prompt(task, tools, history):
    """Build the [system, user] messages for one step of the loop."""
    system = "\n".join([
        "You are a tool-using agent. Solve the task by choosing ONE action at "
        "a time, returned as a single JSON object.",
        "",
        "Available tools:",
        build_tool_descriptions(tools),
        "",
        "To use a tool, reply with exactly:",
        '{"tool": "<name>", "args": {...}}',
        "When you know the final answer, reply with exactly:",
        '{"tool": "finish", "answer": "<answer>"}',
        "",
        "Reply with only the JSON object, nothing else.",
    ])
    lines = ["Task: " + str(task)]
    for step in history:
        lines.append("You called: " + json.dumps(step["action"]))
        lines.append("Result: " + str(step["result"]))
    lines.append("What is your next action?")
    user = "\n".join(lines)
    return [{"role": "system", "content": system},
            {"role": "user", "content": user}]


def run_agent(task, tools=None, llm_fn=None, max_iterations=10):
    """Run the agent loop until it finishes or hits max_iterations.

    The loop is the whole idea of an agent: build a prompt from the task and
    what has happened so far, ask the model for one action, run it, repeat.
    max_iterations is the safeguard - without it a confused model could loop
    forever. Returns:
      {"answer": str, "steps": [...], "iterations": int, "stopped": bool}
    stopped is True if the loop ran out of iterations without finishing.
    """
    if tools is None:
        tools = DEFAULT_TOOLS
    history = []
    for i in range(max_iterations):
        messages = build_agent_prompt(task, tools, history)
        response = call_llm(messages, llm_fn=llm_fn)
        action = parse_action(response)
        if action["type"] == "finish":
            return {"answer": action["answer"], "steps": history,
                    "iterations": i + 1, "stopped": False}
        result = execute_tool(action, tools)
        history.append({"action": action, "result": result})
    return {"answer": "Stopped: reached max_iterations without finishing.",
            "steps": history, "iterations": max_iterations, "stopped": True}


## Automated checks

In [ ]:

score, total = 0, 5
try:
    msgs = build_agent_prompt('add 2 and 2', DEFAULT_TOOLS, [])
    assert isinstance(msgs, list) and msgs[0]['role'] == 'system'
    assert 'calculator' in msgs[0]['content']
    score += 1; print("✅ build_agent_prompt returns system+user messages")

    msgs2 = build_agent_prompt('t', DEFAULT_TOOLS,
                               [{'action': {'tool': 'calculator'}, 'result': '4'}])
    assert '4' in msgs2[1]['content']
    score += 1; print("✅ history is rendered into the user message")

    script = ['{"tool": "calculator", "args": {"expression": "2+2"}}',
              '{"tool": "finish", "answer": "It is 4."}']
    out = run_agent('what is 2+2', DEFAULT_TOOLS, llm_fn=_make_mock_llm(script))
    assert out['answer'] == 'It is 4.' and out['stopped'] is False
    score += 1; print("✅ run_agent loops: tool call, then finish")

    assert len(out['steps']) == 1 and out['steps'][0]['result'] == '4'
    score += 1; print("✅ run_agent records each tool step with its result")

    never = _make_mock_llm(['{"tool": "calculator", "args": {"expression": "1+1"}}'])
    loop = run_agent('x', DEFAULT_TOOLS, llm_fn=never, max_iterations=3)
    assert loop['stopped'] is True and loop['iterations'] == 3
    score += 1; print("✅ max_iterations stops a runaway loop")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── the agent loop ────────────────────────────────────────────────────────────
def build_agent_prompt(task, tools, history):
    """Build the [system, user] messages for one step of the loop."""
    system = "\n".join([
        "You are a tool-using agent. Solve the task by choosing ONE action at "
        "a time, returned as a single JSON object.",
        "",
        "Available tools:",
        build_tool_descriptions(tools),
        "",
        "To use a tool, reply with exactly:",
        '{"tool": "<name>", "args": {...}}',
        "When you know the final answer, reply with exactly:",
        '{"tool": "finish", "answer": "<answer>"}',
        "",
        "Reply with only the JSON object, nothing else.",
    ])
    lines = ["Task: " + str(task)]
    for step in history:
        lines.append("You called: " + json.dumps(step["action"]))
        lines.append("Result: " + str(step["result"]))
    lines.append("What is your next action?")
    user = "\n".join(lines)
    return [{"role": "system", "content": system},
            {"role": "user", "content": user}]


def run_agent(task, tools=None, llm_fn=None, max_iterations=10):
    """Run the agent loop until it finishes or hits max_iterations.

    The loop is the whole idea of an agent: build a prompt from the task and
    what has happened so far, ask the model for one action, run it, repeat.
    max_iterations is the safeguard - without it a confused model could loop
    forever. Returns:
      {"answer": str, "steps": [...], "iterations": int, "stopped": bool}
    stopped is True if the loop ran out of iterations without finishing.
    """
    if tools is None:
        tools = DEFAULT_TOOLS
    history = []
    for i in range(max_iterations):
        messages = build_agent_prompt(task, tools, history)
        response = call_llm(messages, llm_fn=llm_fn)
        action = parse_action(response)
        if action["type"] == "finish":
            return {"answer": action["answer"], "steps": history,
                    "iterations": i + 1, "stopped": False}
        result = execute_tool(action, tools)
        history.append({"action": action, "result": result})
    return {"answer": "Stopped: reached max_iterations without finishing.",
            "steps": history, "iterations": max_iterations, "stopped": True}
```

**Why `for i in range(max_iterations)` and not `while True`?** A model that never emits `finish` would spin forever. Bounding the loop turns a hang into a clean `stopped: True` result — the single most important agent safeguard.

</details>